In [22]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_groq import ChatGroq

from langchain_core.prompts import (
    PromptTemplate,
    FewShotPromptTemplate,
)


In [23]:
# ============================================================
# 2. LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")

In [24]:
# ============================================================
# 3. CONNECT TO NEO4J
# ============================================================

graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

print("Neo4j connection successful!")

Neo4j connection successful!


In [25]:
# ============================================================
# 4. MOVIE DATASET
# ============================================================

movie_query = """
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv'
AS row

MERGE (m:Movie {id: row.movieId})

SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)

FOREACH (director IN split(row.director, '|') |
    MERGE (p:Person {name: trim(director)})
    MERGE (p)-[:DIRECTED]->(m)
)

FOREACH (actor IN split(row.actors, '|') |
    MERGE (p:Person {name: trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m)
)

FOREACH (genre IN split(row.genres, '|') |
    MERGE (g:Genre {name: trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g)
)
"""


# ============================================================
# 5. LOAD DATA
# ============================================================

graph.query(movie_query)

print("Movie dataset loaded successfully.")


# ============================================================
# 6. REFRESH NEO4J SCHEMA
# ============================================================

graph.refresh_schema()

print(graph.schema)

Movie dataset loaded successfully.
Node properties:
Person {age: INTEGER, name: STRING}
Company {name: STRING}
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Person)-[:WORKS_AT]->(:Company)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)


In [26]:
# ============================================================
# 7. CREATE GROQ LLM
# ============================================================

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=0,
)

print("Groq LLM initialized successfully.")


# ============================================================
# 8. TEST GROQ
# ============================================================

response = llm.invoke(
    "Explain Neo4j in one sentence."
)

print(response.content)

Groq LLM initialized successfully.
Neo4j is a high‑performance, ACID‑compliant graph database that stores and queries data as interconnected nodes and relationships, enabling intuitive modeling and fast traversal of complex networks.


In [27]:
# ============================================================
# 9. FEW-SHOT EXAMPLES
# ============================================================

examples = [
    {
        "question": "How many actors are there?",
        "query": """
MATCH (a:Person)-[:ACTED_IN]->(:Movie)
RETURN count(DISTINCT a)
""",
    },

    {
        "question": "Which actors played in the movie Casino?",
        "query": """
MATCH (m:Movie {{title: 'Casino'}})<-[:ACTED_IN]-(a:Person)
RETURN a.name
""",
    },

    {
        "question": "How many movies has Tom Hanks acted in?",
        "query": """
MATCH (a:Person {{name: 'Tom Hanks'}})-[:ACTED_IN]->(m:Movie)
RETURN count(m)
""",
    },

    {
        "question": "List all the genres of the movie Schindler's List",
        "query": """
MATCH (m:Movie {{title: "Schindler's List"}})-[:IN_GENRE]->(g:Genre)
RETURN g.name
""",
    },

    {
        "question": "Which actors have worked in movies from both the comedy and action genres?",
        "query": """
MATCH (a:Person)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g1:Genre),
      (a)-[:ACTED_IN]->(:Movie)-[:IN_GENRE]->(g2:Genre)
WHERE g1.name = 'Comedy'
  AND g2.name = 'Action'
RETURN DISTINCT a.name
""",
    },
]


# ============================================================
# 10. EXAMPLE PROMPT
# ============================================================

example_prompt = PromptTemplate.from_template(
    """User input:
{question}

Cypher query:
{query}
"""
)


In [30]:
# ============================================================
# 11. FEW-SHOT CYPHER PROMPT
# ============================================================

prompt = FewShotPromptTemplate(
    examples=examples,

    example_prompt=example_prompt,

    prefix="""
You are a Neo4j expert.

Your task is to generate a syntactically correct Cypher query
based only on the provided Neo4j schema.

Rules:
- Use only labels, relationships and properties that exist in the schema.
- Do not invent labels.
- Do not invent relationship types.
- Do not invent properties.
- Return only the Cypher query.
- Do not include markdown code fences.
- Use the examples as guidance.

Neo4j schema:

{schema}

Examples:
""",

    suffix="""
User input:
{question}

Cypher query:
""",

    input_variables=["question", "schema"],
)


# ============================================================
# 12. TEST FEW-SHOT PROMPT
# ============================================================

formatted_prompt = prompt.format(
    question="How many actors are there?",
    schema=graph.schema,
)

print(formatted_prompt)


# ============================================================
# 13. CHECK PROMPT VARIABLES
# ============================================================

print("Prompt input variables:")
print(prompt.input_variables)



KeyError: 'title'

In [31]:
# ============================================================
# 14. CREATE GRAPH CYPHER QA CHAIN
# ============================================================

chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,

    cypher_prompt=prompt,

    exclude_types=["Genre"],

    verbose=True,

    return_intermediate_steps=True,

    allow_dangerous_requests=True,
)

print("GraphCypherQAChain created successfully.")

GraphCypherQAChain created successfully.


In [32]:
# ============================================================
# 15. QUESTION 1
# ============================================================

response = chain.invoke({
    "query": "Which actors played in the movie Casino?"
})

print("\nAnswer:")
print(response["result"])


# ============================================================
# 16. QUESTION 2
# ============================================================

response = chain.invoke({
    "query": "How many movies has Tom Hanks acted in?"
})

print("\nAnswer:")
print(response["result"])


# ============================================================
# 17. QUESTION 3
# ============================================================

response = chain.invoke({
    "query": "List all the genres of the movie Schindler's List"
})

print("\nAnswer:")
print(response["result"])


# ============================================================
# 18. QUESTION 4
# ============================================================

response = chain.invoke({
    "query": "Which actors have worked in movies from both the comedy and action genres?"
})

print("\nAnswer:")
print(response["result"])



> Entering new GraphCypherQAChain chain...


KeyError: 'title'

In [33]:
# ============================================================
# 19. VIEW GENERATED CYPHER
# ============================================================

for step in response["intermediate_steps"]:
    print(step)

TypeError: 'AIMessage' object is not subscriptable